# 14 — CSTR simulation, linearization, and parameter estimation

**Objectives.** Validate units and the nominal steady state, compare nonlinear/linear behavior, and estimate heat transfer from noisy observations. State is $x=[C_A,T]$ in kmol/m³ and K; input is jacket temperature; disturbances are feed concentration and temperature.


In [ ]:
import casadi as ca
import numpy as np
from dataclasses import replace
from casadi_tutorial.models.cstr import CSTRParameters,cstr_function,cstr_rk4_function,linearize_cstr,steady_state_residual,cstr_rhs
p=CSTRParameters(); residual=steady_state_residual(p)
print('nominal residual [kmol/m3/h, K/h]:',residual)
assert np.linalg.norm(residual,np.inf) < 2e-4
A,B,E=linearize_cstr(p.nominal_state,p.nominal_input,p.nominal_disturbance,p)
print('A=\n',A,'\neigenvalues=',np.linalg.eigvals(A))


In [ ]:
rng=np.random.default_rng(7); dt=0.02; N=25; jacket=np.full(N,p.nominal_jacket_temperature+5)
step=cstr_rk4_function(dt,p); truth=[p.nominal_state]
for uk in jacket: truth.append(np.asarray(step(truth[-1],[uk],p.nominal_disturbance).full()).ravel())
truth=np.asarray(truth); measured_temperature=truth[:,1]+rng.normal(0,0.03,N+1)
print('temperature range',truth[:,1].min(),truth[:,1].max())


In [ ]:
opti=ca.Opti(); ua=opti.variable(); X=opti.variable(2,N+1)
opti.subject_to(X[:,0]==p.nominal_state); opti.subject_to(opti.bounded(50,ua,300))
estimated=replace(p,heat_transfer=ua)
for k in range(N):
    combined=ca.vertcat(jacket[k],p.nominal_disturbance)
    next_x = X[:,k]
    from casadi_tutorial.integration import rk4_step
    next_x = rk4_step(lambda state,ud:cstr_rhs(state,ud[:1],ud[1:],estimated),X[:,k],combined,dt)
    opti.subject_to(X[:,k+1]==next_x)
opti.minimize(ca.sumsqr((X[1,:]-ca.DM(measured_temperature).T)/0.03))
opti.set_initial(ua,130); opti.set_initial(X,truth.T)
opti.solver('ipopt',{'print_time':False},{'print_level':0})
sol=opti.solve(); print('estimated UA',sol.value(ua),'true UA',p.heat_transfer)
assert abs(sol.value(ua)-p.heat_transfer) < 10


Identifiability is not the same as solver convergence: an uninformative input may give a flat objective even when IPOPT reports success. Inspect parameter sensitivity, confidence under noise, and correlations.

**Common mistakes:** using inconsistent CSTR parameter cases, estimating parameters without bounds, and fitting process noise as physics. **Recap:** validate residuals and simulations before estimation. [Exercises](../exercises/05_cstr.ipynb).
